In [1]:
from datetime import datetime
import time

from google import genai
from google.cloud import bigquery
from google.genai.types import CreateBatchJobConfig
import pandas as pd

In [2]:
from typing import List, Dict, Tuple, Union, Literal, Optional

import pandas as pd
from openai.lib._pydantic import to_strict_json_schema
from pydantic import BaseModel

In [3]:
ROOMS =  ["Kitchen", "Bathroom", "Garden", "Office", "Bedroom", "Hallway", "hello my name is mud"]
CHARS = ["Daniel", "Mary", "Michael", "Sandra", "John"]
NOBODY = "Nobody"

In [4]:
class RoomAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: Literal[*ROOMS]


class NumberAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: str = "0"


class PersonAnswer(BaseModel):
    # reasoning: Optional[str] = None
    answer: Literal[*CHARS, NOBODY] = NOBODY

In [5]:
strict_schemas = {
    "room": {"schema": to_strict_json_schema(RoomAnswer) } | {"name": "room"},
    "number": {"schema": to_strict_json_schema(NumberAnswer) } | {"name": "number"},
    "person": {"schema": to_strict_json_schema(PersonAnswer) } | {"name": "person"},
}

In [6]:
from google.auth import default
import google.auth.transport.requests

project_id = "celtic-bazaar-451314-u6"
location = "us-central1"

In [7]:
import os

if not project_id or project_id == "[your-project-id]":
    project_id = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

In [8]:
client = genai.Client(vertexai=True, project=project_id, location=LOCATION)

In [11]:

BUCKET_URI = "[your-cloud-storage-bucket]"  # @param {type:"string"}

if BUCKET_URI == "[your-cloud-storage-bucket]":
    TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
    BUCKET_URI = f"gs://{project_id}-{TIMESTAMP}"

    ! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

zsh:1: command not found: gsutil


In [12]:
location, BUCKET_URI, project_id

('us-central1',
 'gs://celtic-bazaar-451314-u6-20250307124119',
 'celtic-bazaar-451314-u6')

In [13]:
INPUT_DATA = "gs://cloud-samples-data/generative-ai/batch/batch_requests_for_multimodal_input_2.jsonl" 

In [14]:
MODEL_ID = "gemini-2-flash-001"

In [15]:
gcs_batch_job = client.batches.create(
    model=MODEL_ID,
    src=INPUT_DATA,
    config=CreateBatchJobConfig(dest=BUCKET_URI),
)
gcs_batch_job.name

/home/jovyan/.mlspace/envs/kurkin_exps/lib/python3.12/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


RefreshError: Reauthentication is needed. Please run `gcloud auth application-default login` to reauthenticate.